#Code Cell 1 — Build ColumnTransformer

In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Sample Dataset
df = pd.DataFrame({
    "Age": [22, 25, 23, np.nan, 28, 24, 30, 26],
    "Salary": [25000, 40000, np.nan, 35000, 50000, 30000, 60000, 45000],
    "City": ["Delhi", "Mumbai", "Jaipur", "Delhi", np.nan, "Mumbai", "Jaipur", "Delhi"],
    "Education": ["BTech", "MCA", "BTech", "MBA", "MCA", np.nan, "MBA", "BTech"],
    "Purchased": [0, 1, 0, 1, 1, 0, 1, 1]
})

print("Original Dataset:")
print(df)

# Separate features and target
X = df.drop("Purchased", axis=1)
y = df["Purchased"]

# Identify columns
numeric_features = ["Age", "Salary"]
categorical_features = ["City", "Education"]

# Numerical preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# ColumnTransformer
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

# Fit and transform data
X_processed = preprocessor.fit_transform(X)

# Get transformed feature names
feature_names = preprocessor.get_feature_names_out()

# Convert transformed data into DataFrame
X_processed_df = pd.DataFrame(
    X_processed,
    columns=feature_names
)

print("\nProcessed Dataset:")
print(X_processed_df)

print("\nOriginal Shape:", X.shape)
print("Processed Shape:", X_processed.shape)

print("\nGenerated Feature Names:")
print(feature_names)

Original Dataset:
    Age   Salary    City Education  Purchased
0  22.0  25000.0   Delhi     BTech          0
1  25.0  40000.0  Mumbai       MCA          1
2  23.0      NaN  Jaipur     BTech          0
3   NaN  35000.0   Delhi       MBA          1
4  28.0  50000.0     NaN       MCA          1
5  24.0  30000.0  Mumbai       NaN          0
6  30.0  60000.0  Jaipur       MBA          1
7  26.0  45000.0   Delhi     BTech          1

Processed Dataset:
   num__Age  num__Salary  cat__City_Delhi  cat__City_Jaipur  cat__City_Mumbai  \
0 -1.379636    -1.496711              1.0               0.0               0.0   
1 -0.153293    -0.059868              0.0               0.0               1.0   
2 -0.970855    -0.059868              0.0               1.0               0.0   
3 -0.153293    -0.538816              1.0               0.0               0.0   
4  1.073050     0.898027              1.0               0.0               0.0   
5 -0.562074    -1.017763              0.0               0.0   

#Code Cell 2 — Complete ML Pipeline

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

# Train-test split FIRST
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Complete Pipeline
model_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Train
model_pipeline.fit(X_train, y_train)

# Predict
y_pred = model_pipeline.predict(X_test)

# Evaluate
print("Predictions:", y_pred)
print("Actual:", y_test.values)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nTotal Pipeline Steps:", len(model_pipeline.steps))
print("\nPipeline Structure:")
print(model_pipeline)

Predictions: [1 0]
Actual: [1 0]

Accuracy: 1.0

Confusion Matrix:
[[1 0]
 [0 1]]

Total Pipeline Steps: 2

Pipeline Structure:
Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Salary']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                            